# Importing .py file we saved for reuse simple and easy in one line

In [4]:
import pandas as pd
import sys
sys.path.append('..src/')
from features import build_features

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df_processed = pd.read_csv("../data/processed_customer_churn.csv")
df_model = build_features(df_processed)

print(df_model.shape)
df_model.head()

(7043, 31)


,Senior Citizen,Partner,Tenure,Phone Service,Multiple Lines,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,...,Internet Service_No,Contract_One year,Contract_Two year,Payment Method_Credit card (automatic),Payment Method_Electronic check,Payment Method_Mailed check,tenure_bucket_Loyal (3yr+),tenure_bucket_New (0-1yr),avg_monthly_spend,services_count
0,0,1,1,0,0,0,1,0,0,0,...,False,False,False,False,True,False,False,True,29.85,1
1,0,0,34,1,0,1,0,1,0,0,...,False,True,False,False,False,True,False,False,55.57,2
2,0,0,2,1,0,1,1,0,0,0,...,False,False,False,False,False,True,False,True,54.08,2
3,0,0,45,0,0,1,0,1,1,0,...,False,True,False,False,False,False,True,False,40.91,3
4,0,0,2,1,0,0,0,0,0,0,...,False,False,False,False,True,False,False,True,75.82,0


# Train_test split and scaling

In [6]:

X = df_model.drop(columns=['Churn'])
y = df_model['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_features = ['Tenure', 'Monthly Charges', 'Total Charges', 'Age',
                     'Number of Dependents', 'CLTV', 'Population', 'avg_monthly_spend']

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test_scaled[numeric_features] = scaler.transform(X_test[numeric_features])

print("Train:", X_train_scaled.shape, "Test:", X_test_scaled.shape)

Train: (5634, 30) Test: (1409, 30)


# Model training with random forest as it take care of more complex realtions 

In [7]:

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42,
    max_depth=10
)
rf_model.fit(X_train, y_train)  
print("Random Forest trained.")

Random Forest trained.


In [8]:

from sklearn.metrics import f1_score, roc_auc_score, classification_report, confusion_matrix

y_pred_rf = rf_model.predict(X_test)
y_pred_proba_rf = rf_model.predict_proba(X_test)[:, 1]

f1_rf = f1_score(y_test, y_pred_rf)
roc_auc_rf = roc_auc_score(y_test, y_pred_proba_rf)

print(f"Random Forest F1 Score: {f1_rf:.3f}")
print(f"Random Forest ROC-AUC: {roc_auc_rf:.3f}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred_rf))
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

Random Forest F1 Score: 0.642
Random Forest ROC-AUC: 0.853

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.77      0.83      1035
           1       0.55      0.78      0.64       374

    accuracy                           0.77      1409
   macro avg       0.73      0.77      0.74      1409
weighted avg       0.81      0.77      0.78      1409


Confusion Matrix:
[[792 243]
 [ 82 292]]


In [9]:

from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_cv_f1 = cross_val_score(rf_model, X, y, cv=cv, scoring='f1')
print("Random Forest CV F1 scores:", rf_cv_f1.round(3))
print("Mean:", rf_cv_f1.mean().round(3), "Std:", rf_cv_f1.std().round(3))

Random Forest CV F1 scores: [0.67  0.635 0.664 0.631 0.637]
Mean: 0.647 Std: 0.016


# XG boost 

In [13]:

from xgboost import XGBClassifier

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss'
)
xgb_model.fit(X_train, y_train)
print("XGBoost trained.")

XGBoost trained.


In [14]:

y_pred_xgb = xgb_model.predict(X_test)
y_pred_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

f1_xgb = f1_score(y_test, y_pred_xgb)
roc_auc_xgb = roc_auc_score(y_test, y_pred_proba_xgb)

print(f"XGBoost F1 Score: {f1_xgb:.3f}")
print(f"XGBoost ROC-AUC: {roc_auc_xgb:.3f}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred_xgb))
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))

XGBoost F1 Score: 0.627
XGBoost ROC-AUC: 0.850

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.77      0.83      1035
           1       0.54      0.74      0.63       374

    accuracy                           0.77      1409
   macro avg       0.72      0.76      0.73      1409
weighted avg       0.80      0.77      0.78      1409


Confusion Matrix:
[[802 233]
 [ 97 277]]


In [15]:

xgb_cv_f1 = cross_val_score(xgb_model, X, y, cv=cv, scoring='f1')
print("XGBoost CV F1 scores:", xgb_cv_f1.round(3))
print("Mean:", xgb_cv_f1.mean().round(3), "Std:", xgb_cv_f1.std().round(3))

XGBoost CV F1 scores: [0.64  0.606 0.638 0.623 0.628]
Mean: 0.627 Std: 0.012
